# **Ready Export Data to SQL**

# All Imports Here

In [92]:
import os
from pathlib import Path
import pandas as pd

src_dir  = "d:\\GitHub\\SaaS_Product_Analysis"
# src_dir = Path.cwd()
# os.chdir(src_dir)

# Import Cleaned and Enriched Datasets

In [93]:
zoom_df = pd.read_csv(f'{src_dir}\\data\\cleaned_enriched\\Zoom.csv')
teams_df = pd.read_csv(f'{src_dir}\\data\\cleaned_enriched\\Microsoft Teams.csv')
meet_df = pd.read_csv(f'{src_dir}\\data\\cleaned_enriched\\Google Meet.csv')
webex_df = pd.read_csv(f'{src_dir}\\data\\cleaned_enriched\\Webex.csv')

# Ready Cleaned and Enriched Datasets for Export

In [94]:
zoom_df.head()

,Unnamed: 0,reviewId,rating,body,appVersion,timestamp,sentiment_score,sentiment_label,complaint_score,complaint_flag,category_score,category,category_source,is_feature_request
0,0,0dcdf6d4-dfa0-4401-b3d4-d9dbc27c825f,1,👎👎👎👎👎,NaN,2026-08-26 05:50:54,0.471210,uncertain,0.608053,complaint,0.195305,other,other,True
1,1,d70e1f1c-83fc-495b-8035-3eb460431f82,1,Can't login after resignation,NaN,2026-08-26 05:50:12,0.401677,uncertain,0.605169,complaint,NaN,login / authentication,rule,True
2,2,ad750e18-d0d4-40cc-8797-26e77acdbfe4,1,Worst app,NaN,2026-08-26 05:49:41,0.781650,negative,0.658770,complaint,0.180281,other,other,True
3,3,359d855e-2a8b-4313-8120-91dff1c872ef,1,Can't login,NaN,2026-08-26 05:49:09,0.709888,negative,0.643831,complaint,NaN,login / authentication,rule,False
4,4,dae23cb0-c77e-4f78-89ea-72b01e4c7799,1,Can't login after resignation,NaN,2026-08-26 05:48:36,0.401677,uncertain,0.605169,complaint,NaN,login / authentication,rule,True


In [95]:
'''
curating before export:
    - verify data-type 
'''
def fix_dtypes(df):
    df = df.copy()

    df = df.drop(columns=["Unnamed: 0"], errors="ignore")

    # numeric columns
    int_cols = [
        "rating"
    ]

    float_cols = [
        "sentiment_score",
        "complaint_score",
        "category_score"
    ]

    # boolean columns
    bool_cols = [
        "is_feature_request"
    ]

    # string / categorical text columns
    string_cols = [
        "reviewId",
        "body",
        "appVersion",
        "sentiment_label",
        "complaint_flag",
        "category",
        "category_source"
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    if "is_feature_request" in df.columns:
        df["is_feature_request"] = df["is_feature_request"].astype("boolean")

    # convert timestamp to actual datetime
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

    return df

zoom_df = fix_dtypes(zoom_df)
teams_df = fix_dtypes(teams_df)
meet_df = fix_dtypes(meet_df)
webex_df = fix_dtypes(webex_df)

zoom_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   reviewId            6000 non-null   string        
 1   rating              6000 non-null   Int64         
 2   body                6000 non-null   string        
 3   appVersion          3864 non-null   string        
 4   timestamp           6000 non-null   datetime64[us]
 5   sentiment_score     6000 non-null   float64       
 6   sentiment_label     6000 non-null   string        
 7   complaint_score     6000 non-null   float64       
 8   complaint_flag      6000 non-null   string        
 9   category_score      5211 non-null   float64       
 10  category            6000 non-null   string        
 11  category_source     6000 non-null   string        
 12  is_feature_request  6000 non-null   boolean       
dtypes: Int64(1), boolean(1), datetime64[us](1), float64(3), str

In [96]:
min_dates = [
    min(zoom_df['timestamp']),
    min(meet_df['timestamp']),
    min(teams_df['timestamp']),
    min(webex_df['timestamp']),
]

max_dates = [
    max(zoom_df['timestamp']),
    max(meet_df['timestamp']),
    max(teams_df['timestamp']),
    max(webex_df['timestamp']),
]

min_date = max(min_dates)
max_date = min(max_dates)

In [97]:
cols_to_drop = [
    "Unnamed: 0",
    "appVersion",
    "sentiment_score",
    "complaint_score",
    "category_score",
    "category_source"
]

def df_cleaner(df):
    df = df.drop(columns=cols_to_drop, errors="ignore").copy()

    # make sure timestamp is datetime
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df = df[
        (df["timestamp"] >= min_date) &
        (df["timestamp"] <= max_date)
    ]

    return df


export_zoom = df_cleaner(zoom_df)
export_teams = df_cleaner(teams_df)
export_meet = df_cleaner(meet_df)
export_webex = df_cleaner(webex_df)

In [98]:
'''
- verify data types (ans: done)
- verify and drop duplicate values (Ans: done)
- verify (if possible, remove) NaN values (Ans: done)
'''

export_zoom = export_zoom.drop_duplicates() 
export_meet = export_meet.drop_duplicates() 
export_teams = export_teams.drop_duplicates() 
export_webex = export_webex.drop_duplicates()

In [99]:
export_webex.info()

<class 'pandas.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   reviewId            330 non-null    string        
 1   rating              330 non-null    Int64         
 2   body                330 non-null    string        
 3   timestamp           330 non-null    datetime64[us]
 4   sentiment_label     330 non-null    string        
 5   complaint_flag      330 non-null    string        
 6   category            330 non-null    string        
 7   is_feature_request  330 non-null    boolean       
dtypes: Int64(1), boolean(1), datetime64[us](1), string(5)
memory usage: 19.1 KB


In [100]:
export_webex.isna().sum()

reviewId              0
rating                0
body                  0
timestamp             0
sentiment_label       0
complaint_flag        0
category              0
is_feature_request    0
dtype: int64

# Export for SQL Load

In [101]:
export_zoom.to_csv(f"{src_dir}\\data\\export_data\\zoom.csv")
export_teams.to_csv(f"{src_dir}\\data\\export_data\\teams.csv")
export_meet.to_csv(f"{src_dir}\\data\\export_data\\meet.csv")
export_webex.to_csv(f"{src_dir}\\data\\export_data\\webex.csv")

In [104]:
for df in [export_zoom, export_meet, export_teams, export_webex]:
    print(max(df['timestamp']))

2026-08-25 16:58:53
2026-08-25 18:11:37
2026-08-25 17:49:24
2026-08-25 18:13:04


In [106]:
export_zoom.head()

,reviewId,rating,body,timestamp,sentiment_label,complaint_flag,category,is_feature_request
16,03e3fc3e-fa8b-4a19-bb09-78614460924d,5,Thank you,2026-08-25 16:58:53,positive,uncertain,customer_support,True
17,9d7f788a-35d7-4556-a4f2-5c80d3aa217f,5,good app,2026-08-25 16:46:31,positive,uncertain,other,True
18,a7f1645b-7c65-4058-8edb-e96666d6ba17,5,First time is the charm! It is easy to use & s...,2026-08-25 15:57:19,positive,uncertain,other,True
19,88fe0a9f-a523-4daf-8a99-86041cda9b14,5,zindagi,2026-08-25 15:57:07,uncertain,complaint,other,True
20,31c3ef2b-3d83-452f-a3e2-d66dd4a26e75,5,amazing app,2026-08-25 15:38:05,positive,complaint,other,True
